# Phase 2 — Nettoyer, filtrer et chiffrer l'entonnoir 

In [3]:
import pandas as pd 
donnees = pd.read_csv("../data/numeros_tva.csv",keep_default_na=False)
import sys 
sys.path.append("..")
from validation_structure import normaliser, valider

In [4]:
#Creation d'une colonne avec les numéro tva normaliser
donnees["numero_tva_normalise"]= donnees["numero_tva"].apply(normaliser)
donnees[["numero_tva", "numero_tva_normalise"]].sample(10)

,numero_tva,numero_tva_normalise
9661,DK04344618,DK04344618
2358,XX923899893B09,XX923899893B09
5769,FI48588667,FI48588667
3216,FI-55653615,FI55653615
7757,LU74930372,LU74930372
7921,IT44452408695,IT44452408695
6642,FR38718476612,FR38718476612
6571,IT67874154913,IT67874154913
6211,PT819770086,PT819770086
9614,FR79705170035,FR79705170035


In [5]:
#Création colonne validation_tva (motif)
donnees["validation_tva"] = donnees["numero_tva_normalise"].apply(valider)
donnees[["numero_tva_normalise", "validation_tva"]].sample(10)


,numero_tva_normalise,validation_tva
7635,BE0513701805,"(True, ok)"
9176,BE08076997962,"(False, cle_invalide)"
6973,21255993964,"(False, pays_non_couvert)"
8355,LU3122180O,"(False, cle_invalide)"
2446,LU92774913,"(True, ok)"
7508,NL853760470B49,"(True, ok)"
6461,LU597659247,"(False, cle_invalide)"
1787,PT942114182,"(False, cle_invalide)"
5269,DK98978585,"(True, ok)"
18,BE0605238824,"(True, ok)"


In [6]:
#Nombre pour chaque motif de validation
donnees["validation_tva"].value_counts()

validation_tva
(True, ok)                   6083
(False, cle_invalide)        2605
(False, pays_non_couvert)    1090
(False, vide)                 174
(False, trop_court)            48
Name: count, dtype: int64

- False , vide : l'API ne le trouvera pas car ils ne contiennent rien d'exploitable 
- False, trop court : Ne respecte pas la longeur minimal ( peut etre une erreur de saisie ou être incomplet) on peut les conserver dans le fichiers mais pas essayer sur l'API
- False, pays non couvert : Le module Python ne prend pas en charge ce pays, mais ca ne veut pas dire que le numéro est invalide. Je dois vérifier si le pays est reconnu par l'API avant de l'envoyer.
- False, cle invalide : Ils ne respecte pas les regles de validation à exclure de l'API 
- True , ok : On conserve existance a confirmer avec API

## Doublons

In [7]:
# Nombre de doublons dans donnees
nb_doublons = donnees["numero_tva_normalise"].duplicated().sum()
print("Nombre de doublons : ", nb_doublons)

Nombre de doublons :  696


Un doublon est un numéro de TVA normalisé qui apparaît plusieurs fois, même si les noms des entreprises sont différents. Ces doublons permettront d'éviter des appels inutiles à l'API

In [8]:
#Tableau qui contient uniquement les true ok 
true_ok = donnees[donnees["validation_tva"].str[1] == "ok" ]
len(true_ok)

6083

In [9]:
#Doublons dans le tableau true_ok
true_ok["numero_tva_normalise"].duplicated().sum()
#Supprimer les doublons
true_ok = true_ok.drop_duplicates(subset="numero_tva_normalise")
len(true_ok)

5802

Numéros ayant réussi la validation structurelle : 6 083
Doublons à exclure des appels : 281
Numéros uniques à vérifier avec VIES : 5 802

## Pays non couvert

Le module Python vérifie si le numéro respecte les règles qu'il connaît et s'il ne les connait pas False , pays non couvert. Alors que dans L'API VIES ce même numéro de TVA peut être correcte. 

In [32]:
#Creation tableau pays non couvert
pays_non_couvert = donnees[donnees["validation_tva"].str[1] == "pays_non_couvert"]
len(pays_non_couvert)

1090

In [38]:
#Recuperer le prefixe du numero tva 
pays_non_couvert["numero_tva_normalise"].str[0:2].unique()


<StringArray>
['GB', 'ZZ', 'UK', '58', '39', '38', '33', '88', '10', 'XX',
 ...
 '41', '20', '06', '78', '64', '45', '49', '44', '97', '82']
Length: 105, dtype: str

In [39]:
#Prefixe qui figure dans l'API VIES
pays_vies = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK","EE", "FI", "FR", "DE", "EL", "HU", "IE","IT", "LV", "LT", "LU", "MT", "NL", "PL","PT", "RO", "SK", "SI", "ES", "SE", "XI"]

In [41]:
pays_non_couvert["numero_tva_normalise"].str[0:2].isin(pays_vies).value_counts()

numero_tva_normalise
False    1090
Name: count, dtype: int64

1 090 numéros classés pays_non_couvert ont un préfixe absent de la liste VIES

## Test d'appel de l'API VIES

In [55]:
#Test API 
import requests
url = "https://ec.europa.eu/taxation_customs/vies/rest-api/ms/BE/vat/1997422119"
# url = "https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/27552032534"
DELAI = 10
reponse = requests.get(url, timeout=DELAI)
reponse.status_code
reponse.json()

{'isValid': False,
 'requestDate': '2026-11-17T12:55:53.936Z',
 'userError': 'MS_MAX_CONCURRENT_REQ',
 'name': '---',
 'address': '---',
 'requestIdentifier': '',
 'originalVatNumber': '1997422119',
 'vatNumber': '1997422119',
 'viesApproximate': {'name': '---',
  'street': '---',
  'postalCode': '---',
  'city': '---',
  'companyType': '---',
  'matchName': 3,
  'matchStreet': 3,
  'matchPostalCode': 3,
  'matchCity': 3,
  'matchCompanyType': 3}}

## Temps de réponse de l'API

In [57]:
#Mesurer le temps de réponse
import time
from datetime import datetime

debut = datetime.now()
print(debut)
reponse = requests.get(url, timeout=DELAI)
fin = datetime.now()
print(fin - debut)
print(reponse.json())

2026-11-17 13:58:31.828160
0:00:04.026607
{'isValid': False, 'requestDate': '2026-11-17T12:58:35.804Z', 'userError': 'INVALID', 'name': '', 'address': '', 'requestIdentifier': '', 'originalVatNumber': '1997422119', 'vatNumber': '1997422119', 'viesApproximate': {'name': '---', 'street': '---', 'postalCode': '---', 'city': '---', 'companyType': '---', 'matchName': 3, 'matchStreet': 3, 'matchPostalCode': 3, 'matchCity': 3, 'matchCompanyType': 3}}


In [82]:
#Faire le test avec plusieurs numero de tva 
#liste vide pour stocker les durée et les pays
durees = []
pays = []
#boucle pour essayer plusieur snumero à la suite :
for n in true_ok["numero_tva_normalise"].sample(20):
    url = f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{n[0:2]}/vat/{n[2:]}"
    debut = datetime.now()
    try:
        reponse = requests.get(url, timeout=DELAI)
        print(reponse.json())
        fin = datetime.now()
        durees.append((fin - debut).total_seconds()) # datetime.timedelta(seconds=4, microseconds=342857)en 4.3
        pays.append(n[0:2]) # ajoute dans la liste pays 
    except requests.exceptions.ReadTimeout:
        print("VIES n'a pas répondu à temps")
    time.sleep(10) #10 secondes entre chaques appel
    print(url)
    print(durees)


{'isValid': False, 'requestDate': '2026-11-17T15:36:32.212Z', 'userError': 'INVALID', 'name': '', 'address': '', 'requestIdentifier': '', 'originalVatNumber': '1763177710', 'vatNumber': '1763177710', 'viesApproximate': {'name': '---', 'street': '---', 'postalCode': '---', 'city': '---', 'companyType': '---', 'matchName': 3, 'matchStreet': 3, 'matchPostalCode': 3, 'matchCity': 3, 'matchCompanyType': 3}}
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/BE/vat/1763177710
[3.467171]
{'isValid': False, 'requestDate': '2026-11-17T15:36:42.507Z', 'userError': 'INVALID', 'name': '', 'address': '', 'requestIdentifier': '', 'originalVatNumber': '641447680', 'vatNumber': '641447680', 'viesApproximate': {'name': '---', 'street': '---', 'postalCode': '---', 'city': '---', 'companyType': '---', 'matchName': 3, 'matchStreet': 3, 'matchPostalCode': 3, 'matchCity': 3, 'matchCompanyType': 3}}
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/PT/vat/641447680
[3.467171, 0.283092]
{'isValid': F

In [ ]:
#Faire la moyenne de la durée 

len(durees) #19
#transformer la liste parce que je peux pas utiliser mean sur une liste
durees_pandas = pd.Series(durees).mean()
print(durees_pandas)
print(f"min: {min(durees)} max: {max(durees)}")
ecart_type_duree = pd.Series(durees).std()
print(f" ecrart type: {ecart_type_duree}")

1.1751912631578947
min: 0.106277 max: 7.95989
 ecrart type: 2.056249780379072


## Comparer les temps de réponse par pays

In [85]:
comparaison = pd.DataFrame({ "pays": pays,"durees": durees})
comparaison.groupby("pays").mean()


,durees
pays,
BE,1.868196
FI,0.792071
IT,0.142388
PL,0.540074
PT,0.442893
SE,0.630460


## Estimation du temps nécessaire pour vérifier les 5 802 numéros de TVA

In [ ]:
temps_reponse = len(true_ok) * durees_pandas
temps_pauses = (len(true_ok) - 1) * 10 #time sleep de 10 sec 
temps_total = temps_pauses + temps_reponse
estimation = temps_total / 3600
print(f"Estimaton du temps pour les 5 802 numéros de TVA : {round(estimation,2)} heures")

Estimaton du temps pour les 5 802 numéros de TVA : 18.01 heures


## Estimation du temps necessaire pour les 10 000 numéros de TVA

In [81]:
time_reponse = len(donnees) * durees_pandas
time_pauses = (len(donnees) - 1) * 10
time_total = time_pauses + time_reponse
estimation_time = time_total / 3600
print(f"Estimation du temps pour les 10 000 numéros de TVA : {round(estimation_time,2)} heures")

Estimation du temps pour les 10 000 numéros de TVA : 31.04 heures


Sans nettoyage : 10 000 appels	31,04 heures
Après nettoyage : 5 802 appels	18,01 heures
Temps économisé	13,03 heures

## Conclusion 

### 1. Résultats de la validation structurelle

- **6 083 numéros de TVA** ont réussi la validation structurelle.
- **281 doublons** ont été identifiés et exclus pour éviter des appels inutiles à l'API.
- **5 802 numéros de TVA uniques** restent à vérifier avec VIES.
- **2 605 numéros** ont été rejetés pour une clé invalide (`cle_invalide`).
- **1 090 numéros** ont été classés `pays_non_couvert`, car leur préfixe ne figure pas dans la liste des pays couverts par le module de validation.
- **174 numéros** étaient vides (`vide`).
- **48 numéros** étaient trop courts (`trop_court`).

### 2. Analyse des temps de réponse de l'API VIES

Sur les 20 numéros de TVA testés lors du premier échantillon, **19 temps de réponse ont été enregistrés**.

- **Moyenne :** 1,18 seconde
- **Minimum :** 0,11 seconde
- **Maximum :** 7,96 secondes
- **Écart-type :** 2,06 secondes

Les temps de réponse sont assez dispersés. Certains appels sont très rapides, tandis que d'autres prennent plusieurs secondes.

On observe également une **valeur potentiellement atypique de 7,96 secondes**, car elle est beaucoup plus élevée que la moyenne.

**Comparaison des temps de réponse entre les pays**

Les résultats montrent que les temps de réponse varient selon les pays. Par exemple, la Belgique présente une moyenne de **1,87 seconde**, contre **0,63 seconde pour la Suède**.

Cependant, cet échantillon ne contient que 20 appels répartis entre plusieurs pays. Tous les pays ne sont donc pas représentés et ces résultats ne permettent pas de généraliser les différences observées à l'ensemble des appels VIES.

### 3. Estimation du temps de vérification total

En utilisant le temps de réponse moyen de **1,18 seconde** et en prévoyant une pause de **10 secondes entre chaque appel**, on obtient les estimations suivantes :

- **Sans nettoyage :** la vérification des 10 000 numéros de TVA prendrait environ **31,04 heures**.
- **Après nettoyage :** la vérification des 5 802 numéros de TVA uniques prendrait environ **18,01 heures**.

Le nettoyage des données permet donc d'éviter **4 198 appels à l'API VIES**. Cela représente un gain de temps estimé à **13,03 heures**.


In [ ]:
# Pour la PHASE 3 pour recuperer les doneés netoyées
true_ok.to_csv("../api/true_ok.csv", index=False)